In [56]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score

In [57]:
data_file_path = "data/raw/dailog_acts.dat"

In [58]:
data = []
with open(data_file_path, "r") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue
        
        act, utterance = line.split(maxsplit=1)
        #lowercasing
        data.append({"act": str.lower(act), "utterance": str.lower(utterance)})

In [59]:
df = pd.DataFrame(data=data)

In [60]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23998 entries, 0 to 23997
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   act        23998 non-null  str  
 1   utterance  23998 non-null  str  
dtypes: str(2)
memory usage: 375.1 KB


In [61]:
df.sample(15)

,act,utterance
8482,thankyou,thank you good bye
14455,inform,danish
6582,request,address
19054,inform,i dont care
4176,request,price range
3625,affirm,yes good afternoon im looking for a restaurant...
5003,thankyou,thank you good bye
4935,inform,im looking for a moderately price restaurant t...
18867,request,phone number
19938,null,sil


In [62]:
df["act"].unique()

<StringArray>
[  'inform',   'affirm',  'request', 'thankyou',     'null',      'bye',
  'reqalts',   'negate',  'confirm',    'hello',   'repeat',      'ack',
     'deny',  'restart',  'reqmore']
Length: 15, dtype: str

In [63]:
train_data, test_data = train_test_split(df,test_size=0.15,random_state=42)

In [65]:
with open("data/processed/test/fake_test_data.dat", "w", encoding="utf-8") as f:
    for _, row in test_data.iterrows():
        f.write(f"{row['act']} {row['utterance']}\n")

In [77]:
with open("data/processed/train/fake_train_data.dat", "w", encoding="utf-8") as f:
    for _, row in train_data.iterrows():
        f.write(f"{row['act']} {row['utterance']}\n")

In [66]:
test_data = []
with open("data/processed/test/fake_test_data.dat", "r") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue
        
        act, utterance = line.split(maxsplit=1)
        #lowercasing
        test_data.append({"act": str.lower(act), "utterance": str.lower(utterance)})

In [67]:
test_df =  pd.DataFrame(data=test_data)

In [68]:
test_df.head()

,act,utterance
0,thankyou,noise thank you goodbye
1,reqalts,anything else
2,inform,doesnt matter
3,hello,hi im looking for cuban food
4,inform,i need an expensive restaurant serving british...


In [69]:
word_counts = {'inform' : {},   'affirm': {},  'request': {}, 'thankyou' : {},     'null': {},      'bye': {},
  'reqalts': {},   'negate': {},  'confirm': {},    'hello' : {},   'repeat': {},      'ack': {},
     'deny':{},  'restart':{},  'reqmore':{} }

for _,line in train_data.iterrows():
    act = line["act"]
    words = line["utterance"].split()
    for word in words:
        if word in word_counts[act]:
            word_counts[act][word] += 1
        else:
            word_counts[act][word] = 1

In [70]:
rules = {}

for act, counts in word_counts.items():
    rules[act] = sorted(
        counts,
        key=counts.get,
        reverse=True
)[0]

In [71]:
rules

{'inform': 'food',
 'affirm': 'yes',
 'request': 'the',
 'thankyou': 'thank',
 'null': 'noise',
 'bye': 'good',
 'reqalts': 'about',
 'negate': 'no',
 'confirm': 'it',
 'hello': 'hello',
 'repeat': 'repeat',
 'ack': 'okay',
 'deny': 'wrong',
 'restart': 'start',
 'reqmore': 'more'}

In [72]:
def classify(rules, test_data):
    df = test_data.copy()
    df["pred"] = None

    for index, row in df.iterrows():
        words = row["utterance"].split()
        print(words)

        for act, keyword in rules.items():
            if keyword in words:
                df.loc[index, "pred"] = act
                break
            else:
                df.loc[index, "pred"] = "null"

    return df

In [73]:
pred_df = classify(rules=rules,test_data=test_df)

['noise', 'thank', 'you', 'goodbye']
['anything', 'else']
['doesnt', 'matter']
['hi', 'im', 'looking', 'for', 'cuban', 'food']
['i', 'need', 'an', 'expensive', 'restaurant', 'serving', 'british', 'food']
['anything', 'else']
['thank', 'you', 'goodbye']
['and', 'the', 'post', 'code']
['what', 'is', 'the', 'address', 'phone', 'number', 'and', 'price', 'range']
['what', 'is', 'the', 'address']
['whats', 'the', 'phone', 'number']
['thank', 'you', 'good', 'bye']
['sil']
['what', 'is', 'the', 'post', 'code']
['thank', 'you', 'good', 'bye']
['sea', 'food']
['thank', 'you', 'good', 'bye']
['whats', 'the', 'address']
['restaurant']
['i', 'dont', 'care']
['thai', 'food']
['noise', 'address', 'and', 'the', 'post', 'code', 'of', 'the', 'venue']
['vietnamese']
['italian', 'food']
['im', 'looking', 'for']
['whats', 'the', 'phone', 'number']
['address']
['address']
['what', 'is', 'the', 'phone', 'number']
['what', 'is', 'the', 'phone', 'number']
['thank', 'you', 'good', 'bye']
['thank', 'you', 'good'

In [74]:
pred_df.sample(15)

,act,utterance,pred
1014,request,what is the address and phone number,request
2830,hello,hello,hello
1688,inform,serving jamaican food,inform
1246,inform,north,null
2001,inform,i want chinese food,inform
3488,bye,k you goodbye,null
3089,reqalts,how about in the center,request
484,thankyou,thank you good bye,thankyou
1261,affirm,yes,affirm
2845,inform,expensive restaurant south part of town,null


In [75]:
def evaluate(df):
    y_true = df["act"]
    y_pred = df["pred"]
    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_accuracy:.4f}")

In [76]:
evaluate(pred_df)

Accuracy: 0.5008
Balanced Accuracy: 0.4506
